In [ ]:
import os
import pandas as pd
import seaborn as sns
import numpy as np
import random
from matplotlib import pyplot as plt
from datetime import datetime
%matplotlib inline

import warnings
warnings.filterwarnings('ignore')
from tqdm import tqdm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# 연극

In [ ]:
df = pd.read_excel("/content/drive/MyDrive/캡스톤 디자인/인터파크/연극_TOP70.xlsx")
df=df.drop(['Unnamed: 0'],axis=1)
df=df[['Title','장르']]
df

,Title,장르
0,!로맨틱코미디 연극〈운빨로맨스〉! - 대구,"로맨스, 코믹, 공감/힐링"
1,(리얼타임 코믹연극) 택시안에서 - 부산,"코믹, 로맨스, 공감/힐링"
2,(리얼타임 코믹연극) 택시안에서 - 서울,"코믹, 로맨스, 공감/힐링"
3,(코믹연극) 달동네-부산,"코믹, 감동, 드라마"
4,10년 연속 1위 연극〈옥탑방고양이〉- 틴틴홀,"로맨스, 공감/힐링, 판타지"
...,...,...
65,［서울 대학로］연극 라면,"로맨스, 코미디, 감동"
66,［연극 on stage］ 99% - 화성,"스릴러, 드라마, 미스터리"
67,［연극 on stage］ 띨뿌리 - 화성,"역사극, 미스터리, 사회극"
68,［연애하기 좋은 날 : 당근거래］ - 전주,"로맨스, 코미디"


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

#df 장르에 존재하는 공백 형식이 갖추어지지 않아있어서 공백있는거 공백 제거하기
df['장르']=df['장르'].str.replace(' ', '').str.replace(',', ', ')

#그 다음 ,를 기준으로 장르 나눠주기
df['genres_literal']=df['장르'].str.split(',')
df['genres_literal'] = df['genres_literal'].apply(lambda x : (' ').join(x))
#df
tfidf_vect = TfidfVectorizer()
genre_mat1=tfidf_vect.fit_transform(df['genres_literal'])

print(genre_mat1.shape)

(70, 18)


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

genre_sim1 = cosine_similarity(genre_mat1, genre_mat1)#똑같은 메트릭스를 두개 집어넣는다.
print(genre_sim1.shape)

(70, 70)


In [ ]:
genre_sim_sorted_ind1 = genre_sim1.argsort()[:, ::-1] #내림차순으로 sorting해서 보여줘.
print(genre_sim_sorted_ind1[:1]) #영화 번호들 나옴

[[ 0  1  2 29  4 19 22 21 33 25 41 23 34 20 26 32  9 11 64 68  3 31 15 42
  16 27 28 59 63 60 61 62 18 58 56 57 65 35  8 14 13 12 52 10 53  7 50  6
   5 54 66 67 51 17 36 49 48 47 46 45 44 24 43 40 55 39 30 38 37 69]]


In [ ]:
def find_sim_movie(df, sorted_ind, title_name, top_n=10):
    # 인자로 입력된 movies_df DataFrame에서 'title' 컬럼이 입력된 title_name 값인 DataFrame 추출
    title_movie = df[df['Title'] == title_name]

    # title_name을 가진 DataFrame의 index 객체를 ndarray로 반환하고
    # sorted_ind 인자로 입력된 genre_sim_sorted_ind 객체에서 유사도 순으로 top_n 개의 index 추출
    title_index = title_movie.index.values
    similar_indexes = sorted_ind[title_index, :(top_n + 1)]  # Include one extra for the input movie

    # 추출된 top_n+1 index들 출력. top_n+1 index는 2차원 데이터임.
    # dataframe에서 index로 사용하기 위해서 1차원 array로 변경
    similar_indexes = similar_indexes.reshape(-1)

    # 유사한 영화 중에 유사도가 높은 순으로 정렬한 인덱스에 대응되는 유사도 추출
    similar_scores = genre_sim1[title_index, similar_indexes]

    # DataFrame으로 만들고 유사도가 높은 순으로 정렬
    similar_movie_df = df.iloc[similar_indexes]
    similar_movie_df['Cosine_Similarity'] = similar_scores  # Add cosine similarity scores to the DataFrame

    # Round the cosine similarity values to two decimal places
    similar_movie_df['Cosine_Similarity'] = similar_movie_df['Cosine_Similarity'].round(2)

    similar_movie_df = similar_movie_df[['Title', '장르', 'Cosine_Similarity']]

    # Exclude the input movie from the result
    similar_movie_df = similar_movie_df[similar_movie_df['Title'] != title_name]

    return similar_movie_df.head(top_n)

# Example usage:
similar_movies = find_sim_movie(df, genre_sim_sorted_ind1, '(리얼타임 코믹연극) 택시안에서 - 부산', top_n=10)
similar_movies


,Title,장르,Cosine_Similarity
0,!로맨틱코미디 연극〈운빨로맨스〉! - 대구,"로맨스, 코믹, 공감/힐링",1.00
2,(리얼타임 코믹연극) 택시안에서 - 서울,"코믹, 로맨스, 공감/힐링",1.00
29,연극 〈운빨로맨스〉- 대학로,"로맨스, 코믹, 공감/힐링",1.00
4,10년 연속 1위 연극〈옥탑방고양이〉- 틴틴홀,"로맨스, 공감/힐링, 판타지",0.79
19,뮤직드라마 〈불편한 편의점〉,"드라마, 공감/힐링",0.69
22,연극 〈기형도 플레이〉,"드라마, 공감/힐링",0.69
21,연극 〈극적인 하룻밤〉,"성인용, 로맨스, 공감/힐링",0.69
33,연극 한뼘사이,"로맨스, 코믹",0.60
25,연극 〈뷰티풀라이프〉,"드라마, 감동, 공감/힐링",0.60
41,코믹감동 휴먼판타지 연극 〈2호선 세입자〉,"코믹, 감동, 힐링",0.57


# 뮤지컬

In [ ]:
df = pd.read_excel("/content/drive/MyDrive/캡스톤 디자인/인터파크/뮤지컬_TOP70.xlsx")
df=df.drop(['Unnamed: 0'],axis=1)
df=df[['Title','장르']]
df

,Title,장르
0,"2023 〈스웨그에이지 : 외쳐, 조선〉 - 인천","역사극, 판타지"
1,2023 이은결 더 일루션 마스터피스 - 이천,"마술, 판타지"
2,2023 창작뮤지컬어워드 NEXT,드라마
3,2023 최현우 Answer - 대구,"마술, 판타지"
4,2023 최현우 Answer - 성남,"마술, 판타지"
...,...,...
64,창작가무극 〈순신〉,"역사극, 드라마"
65,태양의서커스 〈루치아〉,"판타지, 액션"
66,［뮤지컬 on stage］ 뮤지컬 〈쇼맨 : 어느 독재자의 네 번째 대역배우〉 - 화성,"드라마, 공감/힐링"
67,［제주］뉴미디어뮤지컬(노형수퍼마켙),액션


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

#df 장르에 존재하는 공백 형식이 갖추어지지 않아있어서 공백있는거 공백 제거하기
df['장르']=df['장르'].str.replace(' ', '').str.replace(',', ', ')

#그 다음 ,를 기준으로 장르 나눠주기
df['genres_literal']=df['장르'].str.split(',')
df['genres_literal'] = df['genres_literal'].apply(lambda x : (' ').join(x))
#df
tfidf_vect = TfidfVectorizer()
genre_mat1=tfidf_vect.fit_transform(df['genres_literal'])

print(genre_mat1.shape)

(69, 15)


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

genre_sim1 = cosine_similarity(genre_mat1, genre_mat1)#똑같은 메트릭스를 두개 집어넣는다.
print(genre_sim1.shape)

(69, 69)


In [ ]:
genre_sim_sorted_ind1 = genre_sim1.argsort()[:, ::-1] #내림차순으로 sorting해서 보여줘.
print(genre_sim_sorted_ind1[:1]) #영화 번호들 나옴

[[ 0 58 23 21 22 31 30 64 59 11 20 27 28 25 24  4  1  6  7  9 10  5 63  3
  65 39 61 47 26 15 19 18 17 16 14 13  8  2 12 68 29 48 66 62 60 57 56 55
  54 53 52 51 50 49 46 32 45 44 43 42 41 40 38 37 36 35 67 33 34]]


In [ ]:
#코사인 유사도 소수점 2째자리까지만
def find_sim_movie(df, sorted_ind, title_name, top_n=10):
    # 인자로 입력된 movies_df DataFrame에서 'title' 컬럼이 입력된 title_name 값인 DataFrame 추출
    title_movie = df[df['Title'] == title_name]

    # title_name을 가진 DataFrame의 index 객체를 ndarray로 반환하고
    # sorted_ind 인자로 입력된 genre_sim_sorted_ind 객체에서 유사도 순으로 top_n 개의 index 추출
    title_index = title_movie.index.values
    similar_indexes = sorted_ind[title_index, :(top_n + 1)]  # Include one extra for the input movie

    # 추출된 top_n+1 index들 출력. top_n+1 index는 2차원 데이터임.
    # dataframe에서 index로 사용하기 위해서 1차원 array로 변경
    similar_indexes = similar_indexes.reshape(-1)

    # 유사한 영화 중에 유사도가 높은 순으로 정렬한 인덱스에 대응되는 유사도 추출
    similar_scores = genre_sim1[title_index, similar_indexes]

    # DataFrame으로 만들고 유사도가 높은 순으로 정렬
    similar_movie_df = df.iloc[similar_indexes]
    similar_movie_df['Cosine_Similarity'] = similar_scores  # Add cosine similarity scores to the DataFrame

    # Round the cosine similarity values to two decimal places
    similar_movie_df['Cosine_Similarity'] = similar_movie_df['Cosine_Similarity'].round(2)

    similar_movie_df = similar_movie_df[['Title', '장르', 'Cosine_Similarity']]

    # Exclude the input movie from the result
    similar_movie_df = similar_movie_df[similar_movie_df['Title'] != title_name]

    return similar_movie_df.head(top_n)

# Example usage:
similar_movies = find_sim_movie(df, genre_sim_sorted_ind1, '태양의서커스 〈루치아〉', top_n=10)
similar_movies


,Title,장르,Cosine_Similarity
67,［제주］뉴미디어뮤지컬(노형수퍼마켙),액션,0.77
58,뮤지컬 〈후크〉,판타지,0.64
13,난타(NANTA) - 홍대공연,"코믹, 액션",0.60
14,넌버벌 코미디 〈옹알스〉,"코믹, 액션",0.60
12,난타(NANTA) - 명동공연,"코믹, 액션",0.60
8,2023송승환의 오리지널 난타(대전),"코믹, 액션",0.60
61,뮤지컬 ＇벤허＇,"드라마, 역사극, 액션",0.55
22,뮤지컬 더데빌 : 에덴 (Musical THE DEVIL : EDEN),"판타지, 드라마",0.53
47,뮤지컬 〈삼총사〉,"역사극, 액션, 코믹",0.51
0,"2023 〈스웨그에이지 : 외쳐, 조선〉 - 인천","역사극, 판타지",0.46
